In [ ]:
import json
from IPython.display import HTML, display

APP = json.loads('"{\\"title\\": \\"Custom Domain Adoption\\", \\"audience\\": \\"Partner researchers, NGOs, and labs evaluating DueCare as a reusable framework rather than a trafficking-only demo.\\", \\"decision\\": \\"Choose this when the question is reusability: can this stack move into a new safety domain cleanly?\\", \\"flow_steps\\": [\\"Choose a new safety domain.\\", \\"Write the pack files that encode taxonomy, rubric, PII, and seed prompts.\\", \\"Register the pack and verify it through the shared harness.\\", \\"Reuse the same evaluation, dashboard, and deployment surfaces.\\"], \\"proof_anchor\\": \\"650 Custom Domain Walkthrough\\"}"')

palette = {
    'primary': '#4c78a8',
    'success': '#10b981',
    'warning': '#f59e0b',
    'info': '#3b82f6',
    'muted': '#6b7280',
    'bg_primary': '#eff6ff',
    'bg_success': '#ecfdf5',
    'bg_warning': '#fffbeb',
    'bg_info': '#eff6ff',
}

def card(label, value, kind='primary'):
    color = palette[kind]
    bg = palette[f'bg_{kind}']
    return (
        f'<div style="display:inline-block;vertical-align:top;width:23%;margin:4px 1%;padding:14px 16px;'
        f'background:{bg};border-left:5px solid {color};border-radius:4px;'
        f'font-family:system-ui,-apple-system,sans-serif">'
        f'<div style="font-size:11px;font-weight:600;color:{color};text-transform:uppercase;letter-spacing:0.04em">{label}</div>'
        f'<div style="font-size:15px;font-weight:600;color:#1f2937;margin-top:6px;line-height:1.35">{value}</div>'
        '</div>'
    )

cards = [
    card('Application', APP['title'], 'primary'),
    card('Primary user', APP['audience'], 'info'),
    card('Use this when', APP['decision'], 'warning'),
    card('Proof anchor', APP['proof_anchor'], 'success'),
]
display(HTML('<div style="margin:10px 0">' + ''.join(cards) + '</div>'))

steps = []
for index, step in enumerate(APP['flow_steps'], start=1):
    steps.append(
        '<div style="display:inline-block;vertical-align:middle;min-width:150px;margin:4px 6px 4px 0;'
        'padding:10px 12px;background:#f8fafc;border:1px solid #d1d5db;border-radius:6px;'
        'font-family:system-ui,-apple-system,sans-serif">'
        f'<div style="font-size:11px;color:#4c78a8;font-weight:600;text-transform:uppercase">Step {index}</div>'
        f'<div style="font-size:13px;color:#1f2937;margin-top:4px">{step}</div>'
        '</div>'
    )

display(HTML(
    '<div style="margin:12px 0 6px 0;font-family:system-ui,-apple-system,sans-serif;font-weight:600;color:#1f2937">'
    'At a glance flow</div>' + ''.join(steps)
))


In [ ]:
# Install the pinned DueCare packages for this notebook.
import glob
import subprocess
import sys

PACKAGES = ['duecare-llm-core==0.1.0']
DUECARE_PACKAGES = [package for package in PACKAGES if package.startswith('duecare-')]
EXTRA_PACKAGES = [package for package in PACKAGES if not package.startswith('duecare-')]

def _pip_install(items):
    if items:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *items])

def _wheel_install():
    wheel_patterns = [
        '/kaggle/input/duecare-llm-wheels/*.whl',
        '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
        '/kaggle/input/**/*.whl',
    ]
    wheel_files = []
    for pattern in wheel_patterns:
        wheel_files = sorted(glob.glob(pattern, recursive=True))
        if wheel_files:
            break
    if not wheel_files:
        return False
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', *wheel_files])
    if EXTRA_PACKAGES:
        _pip_install(EXTRA_PACKAGES)
    print(f'Installed {len(wheel_files)} wheel files via attached Kaggle dataset.')
    return True

def _duecare_importable():
    try:
        import importlib
        for mod in ('duecare.core',):
            importlib.invalidate_caches()
            importlib.import_module(mod)
        return True
    except Exception:
        return False

install_source = 'pypi'
try:
    _pip_install(PACKAGES)
except Exception as exc:
    print(f'Pinned PyPI install failed ({exc.__class__.__name__}). Falling back to Kaggle wheels for DueCare packages.')
    if not _wheel_install() and DUECARE_PACKAGES:
        raise RuntimeError('Could not install pinned DueCare packages from PyPI or attached wheel datasets.') from exc
    install_source = 'kaggle_wheels'
else:
    # PyPI pip install returned 0 but that does not guarantee `duecare` is
    # importable (a stub package with the same name can satisfy pip while
    # providing none of the real modules). Verify; fall back to wheels if
    # the import is still broken.
    if DUECARE_PACKAGES and not _duecare_importable():
        print('PyPI install finished but duecare.core is not importable. Falling back to Kaggle wheels.')
        if _wheel_install():
            install_source = 'kaggle_wheels'
        else:
            raise RuntimeError('duecare.core not importable after pip and wheel install. Attach taylorsamarel/duecare-llm-wheels and rerun.')

import duecare.core
print(f'DueCare version: {duecare.core.__version__} ({install_source})')
if duecare.core.__version__ != '0.1.0':
    print('Expected DueCare version: 0.1.0')


# 695: DueCare Custom Domain Adoption

**Show a partner how the same stack becomes reusable in a new safety domain without changing Python code.**

DueCare is an on-device Gemma 4 safety system. This notebook is not the full implementation deep dive. It is the plain-English deployment playbook for one specific application so a judge or adopter can answer a simple question fast: what does this product do, who uses it, and which existing notebooks prove it is real?

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 22%;">Field</th>
      <th style="padding: 6px 10px; text-align: left; width: 78%;">Value</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;"><b>Inputs</b></td><td style="padding: 6px 10px;">A new domain definition expressed as YAML and JSONL files. The notebook uses the medical-misinformation example as a deployment playbook rather than a code tutorial.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Outputs</b></td><td style="padding: 6px 10px;">A partner-adoption map: what files change, what stays stable, what proof notebooks matter, and how a new domain reaches the same evaluation and deployment surfaces.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Prerequisites</b></td><td style="padding: 6px 10px;">CPU-only narrative notebook. Technical how-to lives in <a href="https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough">650 Custom Domain Walkthrough</a>; generalization proof lives in <a href="https://www.kaggle.com/code/taylorsamarel/duecare-200-cross-domain-proof">200 Cross-Domain Proof</a>.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Runtime</b></td><td style="padding: 6px 10px;">Under 10 seconds. Static adoption playbook only.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Pipeline position</b></td><td style="padding: 6px 10px;">Deployment Applications section. Previous: <a href="https://www.kaggle.com/code/taylorsamarel/duecare-690-migration-case-workflow">690 Migration Case Workflow</a>. Next: <a href="https://www.kaggle.com/code/taylorsamarel/duecare-solution-surfaces-conclusion">899 Solution Surfaces Conclusion</a>. Implementation companions: <a href="https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough">650 Custom Domain Walkthrough</a>, <a href="https://www.kaggle.com/code/taylorsamarel/duecare-200-cross-domain-proof">200 Cross-Domain Proof</a>, <a href="https://www.kaggle.com/code/taylorsamarel/duecare-530-phase3-unsloth-finetune">530 Phase 3 Unsloth Fine-tune</a>.</td></tr>
  </tbody>
</table>


### Why this notebook matters

This is the partner-adoption story in plain English. 650 is the technical walkthrough; this notebook makes the business and deployment meaning of that walkthrough explicit so judges do not have to infer it from YAML alone.

### Reading order

- **Previous notebook:** [690 690 Migration Case Workflow](https://www.kaggle.com/code/taylorsamarel/duecare-690-migration-case-workflow)
- **Next notebook:** [899 899 Solution Surfaces Conclusion](https://www.kaggle.com/code/taylorsamarel/duecare-solution-surfaces-conclusion)
- **Technical how-to:** [650 Custom Domain Walkthrough](https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough)
- **Generalization proof:** [200 Cross-Domain Proof](https://www.kaggle.com/code/taylorsamarel/duecare-200-cross-domain-proof)
- **Model substrate:** [530 Phase 3 Unsloth Fine-tune](https://www.kaggle.com/code/taylorsamarel/duecare-530-phase3-unsloth-finetune)
- **Section close:** [899 Solution Surfaces Conclusion](https://www.kaggle.com/code/taylorsamarel/duecare-solution-surfaces-conclusion)
- **Back to navigation:** [000 Index](https://www.kaggle.com/code/taylorsamarel/duecare-000-index)

### What this notebook does

1. Explains what a partner changes and what stays stable when adopting DueCare.
2. Shows one concrete adoption example without dropping the reader into implementation details too early.
3. Connects framework reuse to the same deployment surfaces the other application notebooks rely on.


## At a glance

The cards below define the application, the primary user, the decision rule for when to deploy it, and the basic operational flow.

## Composite example

This is the simplest believable input and output shape for the application. It is not meant to replace the implementation notebook; it exists so the product story is visible at a glance.

In [ ]:
import json
from IPython.display import HTML, display

PAYLOAD = json.loads('"{\\"sample_input\\": {\\"domain_id\\": \\"medical_misinformation\\", \\"files\\": [\\"taxonomy.yaml\\", \\"rubric.yaml\\", \\"pii_spec.yaml\\", \\"seed_prompts.jsonl\\"], \\"goal\\": \\"Reuse DueCare without modifying Python packages.\\"}, \\"sample_output\\": {\\"adoption_outcome\\": \\"New domain pack discovered by the registry\\", \\"reuse_surfaces\\": [\\"evaluation harness\\", \\"results dashboard\\", \\"API routes\\", \\"deployment playbooks\\"], \\"partner_message\\": \\"Config changes, not framework surgery.\\"}}"')

input_json = json.dumps(PAYLOAD['sample_input'], indent=2)
output_json = json.dumps(PAYLOAD['sample_output'], indent=2)

display(HTML(
    '<table style="width:100%;border-collapse:collapse;margin:4px 0 8px 0;font-family:system-ui,-apple-system,sans-serif">'
    '<thead><tr style="background:#f6f8fa;border-bottom:2px solid #d1d5da">'
    '<th style="padding:8px 10px;text-align:left;width:50%">Composite input</th>'
    '<th style="padding:8px 10px;text-align:left;width:50%">Expected output shape</th>'
    '</tr></thead><tbody><tr>'
    '<td style="padding:8px 10px;vertical-align:top;border-top:1px solid #e5e7eb"><pre style="white-space:pre-wrap;margin:0">' + input_json + '</pre></td>'
    '<td style="padding:8px 10px;vertical-align:top;border-top:1px solid #e5e7eb"><pre style="white-space:pre-wrap;margin:0">' + output_json + '</pre></td>'
    '</tr></tbody></table>'
))


## Repo evidence

The application notebook is only useful if it points back to the notebooks that make the claim defensible.

In [ ]:
import json
from IPython.display import HTML, display

EVIDENCE = json.loads('"{\\"evidence\\": [{\\"notebook\\": \\"650 Custom Domain Walkthrough\\", \\"url\\": \\"https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough\\", \\"proof\\": \\"Shows the exact file-level steps needed to add a new domain pack.\\"}, {\\"notebook\\": \\"200 Cross-Domain Proof\\", \\"url\\": \\"https://www.kaggle.com/code/taylorsamarel/duecare-200-cross-domain-proof\\", \\"proof\\": \\"Shows the shared harness already running across multiple shipped domains.\\"}, {\\"notebook\\": \\"530 Phase 3 Unsloth Fine-tune\\", \\"url\\": \\"https://www.kaggle.com/code/taylorsamarel/duecare-530-phase3-unsloth-finetune\\", \\"proof\\": \\"Shows where domain-specific improvement work plugs into the broader model pipeline.\\"}, {\\"notebook\\": \\"620 Demo API Endpoint Tour\\", \\"url\\": \\"https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour\\", \\"proof\\": \\"Shows that the same deployment surface can sit above multiple domain packs.\\"}]}"')['evidence']

rows = []
for item in EVIDENCE:
    rows.append(
        '<tr>'
        f'<td style="padding:8px 10px;vertical-align:top;border-top:1px solid #e5e7eb"><a href="{item["url"]}">{item["notebook"]}</a></td>'
        f'<td style="padding:8px 10px;vertical-align:top;border-top:1px solid #e5e7eb">{item["proof"]}</td>'
        '</tr>'
    )

display(HTML(
    '<table style="width:100%;border-collapse:collapse;margin:4px 0 8px 0;font-family:system-ui,-apple-system,sans-serif">'
    '<thead><tr style="background:#f6f8fa;border-bottom:2px solid #d1d5da">'
    '<th style="padding:8px 10px;text-align:left;width:30%">Supporting notebook</th>'
    '<th style="padding:8px 10px;text-align:left;width:70%">What it proves</th>'
    '</tr></thead><tbody>' + ''.join(rows) + '</tbody></table>'
))


## Boundaries and handoff

- This notebook is the adoption story, not the file-by-file implementation guide. Use 650 for the exact pack build.
- It explains reuse of the framework, not a guarantee that every domain performs equally without evaluation work.

This notebook is intentionally short. Its job is to make the deployment application legible in minutes, then hand you to the heavier implementation notebook if you want the mechanics.


In [ ]:
print('Custom Domain Adoption handoff >>> 899 899 Solution Surfaces Conclusion https://www.kaggle.com/code/taylorsamarel/duecare-solution-surfaces-conclusion | 600 Results Dashboard https://www.kaggle.com/code/taylorsamarel/600-duecare-results-dashboard | 620 Demo API Endpoint Tour https://www.kaggle.com/code/taylorsamarel/620-duecare-demo-api-endpoint-tour | 650 Custom Domain Walkthrough https://www.kaggle.com/code/taylorsamarel/650-duecare-custom-domain-walkthrough')
